### Random Noise Environment에서 넣을 것

최소 2개.

| 종류                        | 목적    |
| ------------------------- | ----- |
| Gaussian amplitude noise  | 세기 왜곡 |
| Random phase perturbation | 위상 왜곡 |


In [1]:
import os
import sys
from tqdm import tqdm
import torch
import torch.nn as nn
import logging
import csv
from torch.utils.data import DataLoader, random_split

from Utils.dataset import MMFiDataset
from Utils.random_noise import RandomNoisePerturbation

# =====================================================================
# 1. 데이터 로드 및 분활 (CSI & Ground Truth)
# =====================================================================
target_actions = [
    "A01",  # Walking
    "A12",  # Waving arm
    "A08",  # Sitting down
    "A27"   # Falling / bending
]

full_dataset = MMFiDataset(
    env_dirs=["./E01", "./E02", "./E03"],
    target_actions=target_actions,
    perturbation=RandomNoisePerturbation(
        sigma=0.03,
        alpha=0.1
    )
)

# 데이터 샘플 확인 ( 테스트 )
sample_csi, sample_pose = full_dataset[0]

print(sample_csi.shape)
print(sample_pose.shape)


train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0
)

print(f"🔥 Train: {len(train_dataset)}") 
print(f"🔥 Val: {len(val_dataset)}")

# =====================================================================
# 2. 모델 로드 (DT-Pose SOTA)
# =====================================================================
sys.path.append(os.path.abspath("DT-Pose")) 
from model.model import MAE_Encoder, ViT_Pose_Decoder 

encoder = MAE_Encoder( image_size=(114, 30), patch_size=(2, 2), emb_dim=256, input_dim=2 ) 
model = ViT_Pose_Decoder( encoder=encoder, keypoints=17, coor_num=3, dataset='mmfi-csi' ).cuda() 

print("✅ 모델 CUDA 로딩 완료!")

# =====================================================================
# 3. 설정: logging, 체크포인트, CSV 기록 등
# =====================================================================
save_dir = "./checkpoints_fusion"
log_file = os.path.join(save_dir, "randomNoise_training_E01_E03.log")
loss_csv = os.path.join(save_dir, "randomNoise_loss_history_E01_E03.csv")
os.makedirs(save_dir, exist_ok=True)


logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s',
                    handlers=[logging.FileHandler(log_file), logging.StreamHandler()])
logger = logging.getLogger()

# =====================================================================
# 4. Optimizer, Loss Function 설정
# =====================================================================

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4) # 안정성을 위해 1e-4 유지
criterion = nn.MSELoss()

best_val_loss = float('inf')


# =====================================================================
# 5. 초고속 학습 루프 (예외 처리 및 디버깅 로직 포함)
# =====================================================================
logger.info(f"🔥 학습 시작: 총 200 에포크 | Train: {len(train_dataset)} | Val: {len(val_dataset)}")

val_interval = 10 
last_val_loss = float('inf')

for epoch in range(200):
    model.train()
    total_train_loss = 0
    
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:03d}/200", leave=False)
    for batch_idx, (csi_batch, pose_batch) in enumerate(train_pbar):
        try:
            csi_batch, pose_batch = csi_batch.cuda(), pose_batch.cuda()
            
            csi_batch = csi_batch.view(-1, 2, 114, 30) 
            pose_batch = pose_batch.view(-1, 17, 3)
            
            optimizer.zero_grad()
            
            # 모델 출력이 튜플일 경우 첫 번째 요소(예측 포즈) 추출
            output = model(csi_batch)
            predicted_pose = output[0] if isinstance(output, tuple) else output
            predicted_pose = predicted_pose.view(-1, 17, 3)
            
            loss = criterion(predicted_pose, pose_batch)
            
            if torch.isnan(loss):
                raise ValueError(f"NaN Loss 발생! 배치 인덱스: {batch_idx}")
                
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            total_train_loss += loss.item()
            
        except Exception as e:
            logger.error(f"❌ 학습 중 에러 발생 (Epoch {epoch+1}, Batch {batch_idx}): {e}")
            torch.save({'csi': csi_batch, 'pose': pose_batch}, "error_batch.pt")
            print("💾 에러 발생 배치 데이터 'error_batch.pt'로 저장됨.")
            raise e 

    avg_train_loss = total_train_loss / len(train_loader)

    # --- Validation ---
    if epoch == 0 or (epoch + 1) % val_interval == 0:
        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for csi_batch, pose_batch in val_loader:
                csi_batch, pose_batch = csi_batch.cuda(), pose_batch.cuda()
                
                csi_batch = csi_batch.view(-1, 2, 114, 30)
                pose_batch = pose_batch.view(-1, 17, 3)
                
                output = model(csi_batch)
                predicted_pose = output[0] if isinstance(output, tuple) else output
                predicted_pose = predicted_pose.view(-1, 17, 3)
                
                val_loss = criterion(predicted_pose, pose_batch)
                total_val_loss += val_loss.item()
        
        avg_val_loss = total_val_loss / len(val_loader)
        last_val_loss = avg_val_loss 
        
        logger.info(f"Epoch [{epoch+1:03d}/200] | Train Loss: {avg_train_loss:.5f} | Val Loss: {avg_val_loss:.5f}")
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), os.path.join(save_dir, "randomNoise_best_model_E01_E03.pth"))
            logger.info(f"✨ 베스트 모델 갱신 (Val Loss: {avg_val_loss:.5f})")
            
    else:
        logger.info(f"Epoch [{epoch+1:03d}/200] | Train Loss: {avg_train_loss:.5f} | Val Loss: Skipped (Next Val: Epoch {((epoch // val_interval) + 1) * val_interval})")

    with open(loss_csv, 'a', newline='') as f:
        csv.writer(f).writerow([epoch+1, avg_train_loss, last_val_loss])

    if (epoch + 1) % 30 == 0:
        torch.save({'epoch': epoch + 1, 'model_state_dict': model.state_dict()}, 
                   os.path.join(save_dir, f"randomNoise_checkpoint_epoch_E01_E03{epoch+1}.pth"))

logger.info("🎉 학습 완료!")

📂 MMFi Dataset 스캔 시작
▶️ Scan: ./E01
▶️ Scan: ./E02
▶️ Scan: ./E03
✅ 총 샘플 수: 35640
torch.Size([6840])
torch.Size([51])
🔥 Train: 28512
🔥 Val: 7128


c:\Users\yujaerim\miniconda3\envs\deepNet\lib\site-packages\timm\models\layers\__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
2026-06-03 02:45:44,894 [INFO] 🔥 학습 시작: 총 200 에포크 | Train: 28512 | Val: 7128


✅ 모델 CUDA 로딩 완료!


KeyboardInterrupt: 